# AAM in a few Python calls

Match atoms → list unique bond-event candidates → inspect a witness → query a shuffle.

Install once from the repository: `python -m pip install -e ".[notebook]"`. Open this notebook in `docs/` using that Python environment. **Run All uses one CPU and a six-atom example.** No xTB or dataset downloads.

Example: methanol → formaldehyde + H₂. Formal bond orders stand in for WBOs; coordinates are illustrative. Hydrogens are explicit and atom indices start at zero. This is an API demonstration, not a verified mechanism. Display helpers are in [notebook_helpers.py](notebook_helpers.py).

In [1]:
from rxn_core import AAMProblem, AAMSearchConfig, search_aam
from rxn_core.postprocessing import EventDecodeConfig, decode_events
from notebook_helpers import endpoint, show_mapping

problem = AAMProblem(endpoint("CO", "Methanol"),
                     endpoint("C=O.[H][H]", "Formaldehyde + H2"))
config = AAMSearchConfig(iso_tolerance=1.0, seed_count=1,
                         branch_limit=2000, random_seed=42)
aam = search_aam(problem, config, workers=1)  # one direction, cut sweep by default

## Unique candidates (post-processing only)

“Unique” means a signed bond-event pattern modulo the decoder's exact event-response symmetry, including H. It does **not** mean every atom bijection or only the lowest count. The saved AAM remains unchanged. Always inspect `complete`: a timeout leaves useful witnesses but does not prove full coverage of the saved families.

A bond-order increase such as C–O → C=O counts as a positive event, even though the atoms were already bonded.

In [2]:
decoded = decode_events(aam, EventDecodeConfig(threshold=0.5, metal_threshold=0.3))
candidates = decoded.candidates
print("Complete saved-family decoding:", decoded.complete)
for i, candidate in enumerate(candidates):
    print(i, "events:", candidate.total, candidate.events, "mapping:", candidate.mapping)
assert decoded.complete
assert [c.total for c in candidates] == [4, 6]

Complete saved-family decoding: True
0 events: 4 {'broken': ((0, 4), (1, 5)), 'formed': ((0, 1), (4, 5))} mapping: {0: 0, 1: 1, 2: 4, 3: 5, 4: 3, 5: 2}
1 events: 6 {'broken': ((0, 3), (0, 4), (1, 5)), 'formed': ((0, 1), (0, 5), (3, 4))} mapping: {0: 0, 1: 1, 2: 5, 3: 3, 4: 2, 5: 4}


## Inspect one witness

Change `choice` to inspect another event class. The two panes use original atom indices; matched atoms share a color. The stick display loads [3Dmol.js](https://github.com/3dmol/3Dmol.js/tree/master/py3Dmol), so a trusted notebook and internet access are needed for the interactive view.

In [3]:
choice = 0
candidate = candidates[choice]
witness = candidate.mapping
show_mapping(problem, witness).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## What symmetry is still available?

A unique witness retains links to compressed families and their provenance. `group` entries contain generators; `pool` entries describe interchangeable target pools. Their order and preservation constraints matter. These are search actions, **not permission to swap each atom independently**.

In [4]:
symmetry = decoded.symmetry(candidate)
print("Supporting families:", len(symmetry))
print("First family actions:", symmetry[0]["actions"])
print("Required fragment edges:", symmetry[0]["required_edges"])

Supporting families: 5
First family actions: (('pool', (2, 3)), ('group', ((0, 1, 3, 2, 4, 5),)), ('group', ((0, 1, 2, 3, 5, 4), (0, 1, 3, 2, 4, 5))))
Required fragment edges: ((0, 1), (0, 2), (0, 3))


## Ask whether a shuffle is allowed

Swap the target images of methanol H2 and H3, keeping all other assignments fixed. The query checks joint family constraints **and the same event class**. It returns `allowed`, `forbidden` within the saved families, or `unknown` if its budget expires. No full permutation list is generated.

In [5]:
proposed = dict(witness)
proposed[2], proposed[3] = proposed[3], proposed[2]
shuffle = decoded.query(candidate, proposed, same_events=True)
print(shuffle.status, shuffle.mapping)
assert shuffle.status == "allowed"
show_mapping(problem, shuffle.mapping).show()

allowed {0: 0, 1: 1, 2: 5, 3: 4, 4: 3, 5: 2}


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Passing only some assignments lets other atoms move jointly. `same_events=False` asks about the broader saved AAM relation and may return another event class. This separates broad matching (`iso_tolerance`) from final event thresholds.

In [6]:
conditional = decoded.query(candidate, {2: proposed[2]}, same_events=True)
print("Partial condition:", conditional.status)
other = candidates[1 if choice == 0 else 0]
print("Other class, preserve events:", decoded.query(candidate, other.mapping).status)
print("Other class, any saved events:", decoded.query(candidate, other.mapping, same_events=False).status)
# Reuse exactly the same AAM with another event definition:
coarser = decode_events(aam, EventDecodeConfig(threshold=1.1, metal_threshold=None))
print("At threshold 1.1:", [c.total for c in coarser.candidates])

Partial condition: allowed
Other class, preserve events: forbidden
Other class, any saved events: allowed
At threshold 1.1: [0]


## Optional controls: anchors, directions, conditional matching

Anchors use input R→P indices. Bidirectional search retains two directed compressed results; invert concrete witnesses with `to_input_mapping()`. Do not invert generator arrays or compare direction-local event IDs as if they shared one canonical frame.

In [7]:
from dataclasses import replace
from rxn_core import search_aam_directions

anchored = replace(config, anchors=((0, 0),))  # require carbon 0 -> carbon 0
runs = search_aam_directions(problem, anchored, direction="both", workers=1)
for run in runs:
    path = next(run.aam.graph.paths())
    print(run.plan.direction, run.to_input_mapping(path.mapping))
# Also available: direction="forward", "reverse", "smaller_first", "larger_first".

R_to_P {0: 0, 1: 1, 2: 4, 3: 5, 4: 3, 5: 2}
P_to_R {0: 0, 1: 1, 5: 2, 2: 3, 3: 4, 4: 5}


In [8]:
from rxn_core import build_graph, match_fragment, FragmentMatchConfig, FragmentMatchContext

source = build_graph(problem.reactant.elements, problem.reactant.wbo, bond_cut=0.2)
target = build_graph(problem.product.elements, problem.product.wbo, bond_cut=0.2)
fragment = match_fragment(source, target, seed=1,
    context=FragmentMatchContext(locked_mapping={0: 0}),
    config=FragmentMatchConfig(iso_tolerance=1.0, branch_limit=2000))
print("Capped:", fragment.capped)
print("Conditional placements:", [dict(m) for m in fragment.matches])

Capped: False
Conditional placements: [{1: 1, 0: 0, 3: 4, 2: 5}]


For fixed-query isomorphism, use `rxn_core.match_weighted_subgraph(..., anchor_map=..., iso_tol=...)`.

The published default is `search_aam(...)` followed by `decode_events(aam, ...)`. Fragment competition remains an experimental, explicit opt-in extension and is off by default; see [the Python API guide](PYTHON_API.md).

**Chirality TODO:** raw AAM and this event decoder do not guarantee stereochemical preservation. `select_rp_mappings` has downstream coordinate/index-chirality filtering; that is not a general CIP/E–Z-aware AAM contract.

See [PYTHON_API.md](PYTHON_API.md) for all controls, partial mappings, checkpointing, native installation, and the software audit.